# AIHub T자형 교차로 데이터 점검 03 — 중복 그룹과 대표 영상 결정

실용적인 정리 기준으로 pHash 평균 Hamming 거리 `12 이하`인 영상들을 같은 그룹으로 연결하고, 그룹마다 화질이 가장 좋은 영상 하나를 대표로 선택합니다.

원본 영상은 삭제하거나 이동하지 않습니다. 결과는 `keep` 열이 포함된 manifest CSV로만 저장합니다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

REPO_ROOT = Path('/home/sungmin/blackbox-dacon')
METADATA_DIR = REPO_ROOT / 'data/stage2/aihub_tjunction/metadata'
INVENTORY_PATH = METADATA_DIR / 'inventory.csv'
FINGERPRINT_PATH = METADATA_DIR / 'brightness_fingerprints_v1.npz'
QUALITY_PATH = METADATA_DIR / 'video_quality_v1.csv'
MANIFEST_PATH = METADATA_DIR / 'dedup_manifest_threshold12.csv'

PHASH_THRESHOLD = 12.0

inventory = pd.read_csv(INVENTORY_PATH)
quality = pd.read_csv(QUALITY_PATH)
cache = np.load(FINGERPRINT_PATH, allow_pickle=True)
fingerprints = cache['fingerprints'].astype(bool)
cached_video_ids = cache['video_ids'].astype(str)

assert len(inventory) == len(quality) == len(fingerprints)
assert np.array_equal(cached_video_ids, inventory['video_id'].astype(str).to_numpy())
assert np.array_equal(quality['video_id'].astype(str), inventory['video_id'].astype(str))

print(f'전체 영상: {len(inventory):,}개')
print(f'pHash 그룹 기준: {PHASH_THRESHOLD:g} 이하')

## 거리 기준으로 연결 요소 만들기

A-B와 B-C가 기준 이하라면 A, B, C를 같은 그룹으로 취급합니다. 이 결정은 라벨링용 데이터 수를 빠르게 줄이기 위한 보수적인 정리이며 원본에는 영향을 주지 않습니다.

In [ ]:
class UnionFind:
    def __init__(self, size: int):
        self.parent = np.arange(size)
        self.rank = np.zeros(size, dtype=np.int16)

    def find(self, value: int) -> int:
        while self.parent[value] != value:
            self.parent[value] = self.parent[self.parent[value]]
            value = int(self.parent[value])
        return value

    def union(self, left: int, right: int):
        left_root, right_root = self.find(left), self.find(right)
        if left_root == right_root:
            return
        if self.rank[left_root] < self.rank[right_root]:
            left_root, right_root = right_root, left_root
        self.parent[right_root] = left_root
        if self.rank[left_root] == self.rank[right_root]:
            self.rank[left_root] += 1


video_count, sample_count, _ = fingerprints.shape
union_find = UnionFind(video_count)
candidate_edge_count = 0

for left_index in tqdm(range(video_count - 1), desc='중복 그룹 생성'):
    different_bits = np.count_nonzero(
        fingerprints[left_index + 1:] != fingerprints[left_index],
        axis=(1, 2),
    )
    distances = different_bits / sample_count
    matched_offsets = np.flatnonzero(distances <= PHASH_THRESHOLD)
    for offset in matched_offsets:
        union_find.union(left_index, left_index + 1 + int(offset))
    candidate_edge_count += len(matched_offsets)

roots = np.array([union_find.find(index) for index in range(video_count)])
root_order = sorted(np.unique(roots), key=lambda root: int(np.flatnonzero(roots == root)[0]))
root_to_group = {root: f'group_{number:04d}' for number, root in enumerate(root_order, start=1)}
group_ids = np.array([root_to_group[root] for root in roots])

print(f'기준 이하 영상 쌍: {candidate_edge_count:,}쌍')
print(f'생성된 그룹: {len(root_order):,}개')

## 그룹마다 대표 영상 하나 선택하기

검거나 하얗게 뭉개진 픽셀 비율이 가장 낮은 영상을 먼저 선택합니다. 같다면 선명도, 해상도, 파일 크기가 큰 영상을 우선합니다.

In [ ]:
manifest = inventory.merge(quality, on='video_id', how='left', validate='one_to_one')
manifest['group_id'] = group_ids
manifest['group_size'] = manifest.groupby('group_id')['video_id'].transform('size')
manifest['clip_ratio'] = manifest['dark_clip_ratio'] + manifest['bright_clip_ratio']
manifest['pixel_count'] = manifest['width'] * manifest['height']

quality_order = manifest.sort_values(
    ['group_id', 'clip_ratio', 'sharpness', 'pixel_count', 'size_mb', 'video_id'],
    ascending=[True, True, False, False, False, True],
)
representative_indices = quality_order.groupby('group_id', sort=False).head(1).index
manifest['keep'] = False
manifest.loc[representative_indices, 'keep'] = True
manifest['representative_video_id'] = manifest['group_id'].map(
    manifest.loc[representative_indices].set_index('group_id')['video_id']
)
manifest['is_duplicate_group'] = manifest['group_size'] > 1
manifest['phash_threshold'] = PHASH_THRESHOLD

output_columns = [
    'video_id', 'file_name', 'relative_path',
    'group_id', 'group_size', 'is_duplicate_group',
    'keep', 'representative_video_id', 'phash_threshold',
    'size_mb', 'frame_count', 'fps', 'duration_sec', 'width', 'height',
    'brightness_mean', 'dark_clip_ratio', 'bright_clip_ratio', 'sharpness',
]
manifest[output_columns].to_csv(MANIFEST_PATH, index=False, encoding='utf-8-sig')

kept_count = int(manifest['keep'].sum())
duplicate_group_count = int(
    manifest.loc[manifest['is_duplicate_group'], 'group_id'].nunique()
)
summary = pd.Series({
    '원본 영상': len(manifest),
    '대표 영상으로 유지': kept_count,
    '대표에서 제외': len(manifest) - kept_count,
    '중복 후보 그룹': duplicate_group_count,
    '가장 큰 그룹 크기': int(manifest['group_size'].max()),
})

print('manifest 저장:', MANIFEST_PATH)
display(summary.to_frame('값'))

In [ ]:
print('그룹 크기 분포')
display(
    manifest.groupby('group_id').size().value_counts().sort_index().rename('그룹 수').to_frame()
)

print('가장 큰 중복 후보 그룹')
largest_group_ids = (
    manifest.groupby('group_id').size().sort_values(ascending=False).head(10).index
)
largest_group_columns = [
    'group_id', 'video_id', 'group_size', 'keep',
    'brightness_mean', 'clip_ratio', 'sharpness',
]
display(
    manifest.loc[
        manifest['group_id'].isin(largest_group_ids),
        largest_group_columns,
    ].sort_values(
        ['group_size', 'group_id', 'keep'],
        ascending=[False, True, False],
    )
)

## 이번 노트북의 종료 지점

`dedup_manifest_threshold12.csv`의 `keep=True`인 영상만 이후 필터링과 라벨링 대상으로 사용합니다. 원본 MP4는 그대로 유지합니다. 다음 단계에서 이 manifest를 이용해 영상 내용 필터링과 Stage 2 라벨링을 한 번에 진행합니다.